# Замеры LinQ (P58)
Загрузите в папку ноутбука и нажмите **Run All**:
- `packages/` с `ppe.tpu`, `people.tpu`, `pose.tpu` и парными `*.tpu.json`;
- `ml.zip` (из репо: `git archive -o ml.zip HEAD ml`), если рядом нет папки `ml/`;
- по желанию: `fp32/ppe.onnx` и `val/` (картинки среза val СИЗ, разметка YOLO рядом в `labels/`) для сравнения INT8 и FP32; `videos/` с роликами для режима воркера.

Итог: `linq_bench_result.json` (ссылка на скачивание), `events.jsonl` и `events.json` (для `demo/replay_events.py --file events.json`).

In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path

import cv2
import numpy as np

HERE = Path.cwd()
PACKAGES = Path(os.environ.get("LINQ_PACKAGES", "packages"))
FP32_ONNX = Path(os.environ.get("LINQ_FP32_PPE", "fp32/ppe.onnx"))
VAL_DIR = Path(os.environ.get("LINQ_VAL", "val"))
VIDEOS_DIR = Path(os.environ.get("LINQ_VIDEOS", "videos"))
FRAMES = int(os.environ.get("LINQ_FRAMES", "50"))  # кадров на поток в замере
CAMERA_FPS = float(os.environ.get("LINQ_CAMERA_FPS", "5"))  # кадров/с на камеру для расчёта ёмкости
WORKER_FRAMES = int(os.environ.get("LINQ_WORKER_FRAMES", "300"))


# найти пакеты проекта: папка ml/ рядом или ml.zip
def _find_ml(start):
    for base in (start, *start.parents):
        for cand in (base, base / "ml"):
            if (cand / "worker" / "infer.py").is_file() and (cand / "export").is_dir():
                return cand
    import zipfile

    arch = next((p for p in (start / "ml.zip", start.parent / "ml.zip") if p.is_file()), None)
    if arch is None:
        raise FileNotFoundError("нет папки ml/ и ml.zip: загрузите ml.zip")
    zipfile.ZipFile(arch).extractall(start / "_ml")
    return _find_ml(start / "_ml")


try:  # пакеты уже доступны (запуск из репозитория)
    if importlib.util.find_spec("worker") is None or importlib.util.find_spec("export") is None:
        raise ImportError
    ML_DIR = Path(importlib.util.find_spec("worker").origin).resolve().parents[1]
except ImportError:
    ML_DIR = _find_ml(HERE)
    sys.path.insert(0, str(ML_DIR))
from export import linq_bench as lb  # noqa: E402

print("ml:", ML_DIR)

In [ ]:
# Устройство LinQ: без pytpu или без устройств дальше идти нельзя
devices = lb.check_linq()
print("устройства LinQ:", devices)
result = {"devices": devices, "camera_fps": CAMERA_FPS}

In [ ]:
programs = lb.find_programs(PACKAGES)
missing = [r for r in lb.ROLES if r not in programs]
print("программы:", programs)
if not programs:
    raise FileNotFoundError(f"в {PACKAGES} нет пар *.tpu + *.tpu.json")
if missing:
    print("НЕТ программ:", missing)
result["programs_missing"] = missing

In [ ]:
# Кадр для замера: первая картинка val, иначе пустой 720p
val_images = lb.list_val(VAL_DIR) if VAL_DIR.is_dir() else []
frame = np.zeros((720, 1280, 3), dtype=np.uint8)
if val_images:
    img = cv2.imread(str(val_images[0]))
    frame = img if img is not None else frame
print("кадр:", frame.shape)

In [ ]:
# FPS, p50/p95 инференса, пред- и постобработка; 1, 2 и 4 потока; каждая программа и все вместе
bench = lb.bench_programs(programs, frame, frames=FRAMES, camera_fps=CAMERA_FPS)
result.update(bench)
print("\n".join(lb.table(bench)))

In [ ]:
# Ёмкость: сколько камер по CAMERA_FPS кадров/с тянет связка СИЗ + люди + поза на кадр
if bench["combined"]:
    for run in bench["combined"]:
        print(
            f"{run['streams']} поток(ов): {run['fps']:.1f} кадров/с -> "
            f"{run['cameras']} камер по {CAMERA_FPS:g} кадров/с"
        )
else:
    print("вместе не замерено: меньше двух программ")

In [ ]:
# Точность INT8 (.tpu) против FP32 (.onnx) на срезе val СИЗ
result["int8_vs_fp32"] = {"skipped": None}
if "ppe" not in programs:
    result["int8_vs_fp32"]["skipped"] = "нет программы ppe"
elif not FP32_ONNX.is_file():
    result["int8_vs_fp32"]["skipped"] = f"нет {FP32_ONNX}"
elif not val_images:
    result["int8_vs_fp32"]["skipped"] = f"нет картинок в {VAL_DIR}"
else:
    try:
        import onnxruntime as ort

        from worker.infer import LinqSession

        fp32 = ort.InferenceSession(str(FP32_ONNX), providers=["CPUExecutionProvider"])
        cmp = lb.compare_int8_fp32(LinqSession(programs["ppe"]), fp32, val_images)
        result["int8_vs_fp32"] = cmp
    except ImportError as exc:
        result["int8_vs_fp32"]["skipped"] = f"нет onnxruntime: {exc}"
print(json.dumps(result["int8_vs_fp32"], ensure_ascii=False, indent=2))

In [ ]:
# Режим воркера: ролики -> воркер на LinQ -> события в JSONL (и в JSON-массив для replay_events.py)
videos = lb.list_videos(VIDEOS_DIR) if VIDEOS_DIR.is_dir() else []
result["worker"] = {"videos": videos, "events": None, "skipped": None}
if not videos:
    result["worker"]["skipped"] = f"нет роликов в {VIDEOS_DIR}"
elif "people" not in programs:
    result["worker"]["skipped"] = "нет программы people"
else:
    n = lb.run_worker(programs, videos, "events.jsonl", max_frames=WORKER_FRAMES)
    lb.jsonl_to_array("events.jsonl", "events.json")
    result["worker"]["events"] = n
print(result["worker"])

In [ ]:
# Итог: JSON для скачивания
from IPython.display import FileLink, display

out = lb.write_result(result, "linq_bench_result.json")
display(FileLink(out))
for extra in ("events.jsonl", "events.json"):
    if Path(extra).is_file():
        display(FileLink(extra))

In [ ]:
# По желанию: положить итог на другой Jupyter (LINQ_JUPYTER_URL); логин из LINQ_JUPYTER_USER /
# LINQ_JUPYTER_PASSWORD или вводом, в ноутбуке секретов нет
url = os.environ.get("LINQ_JUPYTER_URL", "")
if url:
    user, secret = lb.jupyter_login()
    data = Path(out).read_bytes()
    print("статус выгрузки:", lb.upload_result(url, "linq_bench_result.json", data, secret))
else:
    print("LINQ_JUPYTER_URL не задан: выгрузка пропущена")